In [2]:
import cv2
import json
import time
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
import os


## 1. CONFIGURATION
## ==========================================

In [3]:
MODEL_PATH = 'best_engagement_model.keras'
MAPPING_PATH = 'class_mapping.json'
IMG_SIZE = (96, 96) # Must match train_pipeline.py


## 2. LOAD ASSETS
## ==========================================

In [4]:
if not os.path.exists(MODEL_PATH) or not os.path.exists(MAPPING_PATH):
    raise FileNotFoundError("Model or mapping not found! Please run train_pipeline.py first.")

print("Loading model...")
model = load_model(MODEL_PATH)

print("Loading class mapping...")
with open(MAPPING_PATH, 'r') as f:
    mapping = json.load(f)
    class_names = [mapping[str(i)] for i in range(len(mapping))]

# Initialize face detector
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')


Loading model...
Loading class mapping...


## 3. REALTIME DETECTION
## ==========================================

In [5]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise IOError("Cannot open webcam")

print("Starting webcam... Press 'q' to quit.")

# Variables for FPS calculation
prev_time = 0
fps = 0

while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame")
        break
        
    current_time = time.time()
    if current_time - prev_time > 1.0:
        # Calculate FPS over 1 second intervals for printing
        # Actually it's better to just do it per frame but print occasionally
        pass
        
    # Calculate instant FPS
    fps = 1 / (current_time - prev_time) if prev_time > 0 else 0
    prev_time = current_time

    # Preprocessing
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # Detect faces
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))
    
    for (x, y, w, h) in faces:
        # Crop face
        face_img = frame[y:y+h, x:x+w]
        
        # Preprocess to match training
        face_resized = cv2.resize(face_img, IMG_SIZE)
        face_rgb = cv2.cvtColor(face_resized, cv2.COLOR_BGR2RGB)
        face_norm = face_rgb.astype('float32') / 255.0
        face_batch = np.expand_dims(face_norm, axis=0)
        
        # Inference
        preds = model.predict(face_batch, verbose=0)[0]
        class_idx = np.argmax(preds)
        confidence = preds[class_idx]
        class_name = class_names[class_idx]
        
        # Draw bounding box and label
        color = (0, 255, 0) if "Engaged" in class_name or class_name in ['Focused'] else (0, 0, 255)
        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
        
        label = f"{class_name} ({confidence:.2f})"
        cv2.putText(frame, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    
    # Display FPS on frame
    cv2.putText(frame, f"FPS: {fps:.1f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
    
    # Show output
    cv2.imshow('Real-time Engagement Detection', frame)
    
    # Occasionally print FPS to console as requested
    if np.random.rand() < 0.05:
        print(f"Current FPS: {fps:.1f}")
        
    # Quit condition
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("Quit successfully.")


Starting webcam... Press 'q' to quit.
Current FPS: 16.2
Current FPS: 16.0
Current FPS: 15.9
Current FPS: 16.3
Current FPS: 16.2
Current FPS: 16.1
Current FPS: 16.2
Current FPS: 16.1
Current FPS: 16.1
Current FPS: 16.2
Current FPS: 16.6
Current FPS: 31.4
Current FPS: 13.0
Current FPS: 16.1
Current FPS: 16.1
Current FPS: 16.2
Current FPS: 16.2
Current FPS: 15.4
Current FPS: 16.2
Current FPS: 15.9
Current FPS: 16.0
Current FPS: 16.7
Current FPS: 16.2
Current FPS: 32.1
Current FPS: 9.0
Current FPS: 15.9
Current FPS: 15.9
Current FPS: 16.1
Current FPS: 16.1
Current FPS: 16.1
Current FPS: 16.1
Current FPS: 15.8
Current FPS: 15.9
Current FPS: 16.0
Current FPS: 15.9
Current FPS: 15.9
Current FPS: 15.9
Current FPS: 16.0
Current FPS: 31.2
Current FPS: 16.1
Current FPS: 16.1
Current FPS: 15.8
Current FPS: 16.1
Current FPS: 15.6
Current FPS: 16.2
Current FPS: 12.9
Current FPS: 12.9
Current FPS: 15.9
Quit successfully.
